# Karat Trend — Signal Lab

Python re-implementation of the signal logic in `KaratTrend_Divergence_EA.mq5` (v1.10), using this
desk's exact parameter preset (4H, Swing trading — moderate frequency).

**Scope:** reproduces the *signal formulas* -- RSI, Stochastic, ATR, EMA, fractal pivot
detection, regular/hidden divergence, and the price-action confirmation candle -- plus
a simplified backtest loop mirroring the EA's entry/exit rules.

**Not modeled:** broker spread/slippage precision, partial fills, margin, multi-position
netting, or MT5's exact tick-by-tick execution. Use for signal research, then confirm
any promising setup with the real MT5 Strategy Tester (see `BACKTEST.md`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_rows', 20)
%matplotlib inline

## 1. Load OHLC data

Expects a CSV with columns `time, open, high, low, close` (export from MT5: right-click
chart -> Save As). A small synthetic dataset is generated as a fallback so the notebook
runs end-to-end without any data of your own.

In [ ]:
DATA_PATH = "xauusd.csv"  # <-- point this at your exported MT5 history

def load_data(path):
    try:
        d = pd.read_csv(path, parse_dates=["time"])
        d = d.sort_values("time").reset_index(drop=True)
        return d[["time", "open", "high", "low", "close"]]
    except FileNotFoundError:
        print(f"'{path}' not found -- generating synthetic OHLC data instead so the notebook still runs.")
        return generate_synthetic_ohlc()

def generate_synthetic_ohlc(n=3000, seed=7):
    rng = np.random.default_rng(seed)
    dt = pd.date_range("2023-01-01", periods=n, freq="1h")
    steps = rng.normal(0, 3.5, n)
    trend = np.sin(np.linspace(0, 14, n)) * 1.2
    close = 1900 + np.cumsum(steps + trend)
    high = close + rng.uniform(0.5, 5, n)
    low = close - rng.uniform(0.5, 5, n)
    open_ = close - rng.normal(0, 2.5, n)
    high = np.maximum.reduce([high, open_, close])
    low = np.minimum.reduce([low, open_, close])
    return pd.DataFrame({"time": dt, "open": open_, "high": high, "low": low, "close": close})

df = load_data(DATA_PATH)
df.tail()

## 2. Indicator formulas (RSI, Stochastic, ATR, EMA)

Match MT5's default `iRSI`, `iStochastic`, `iATR`, `iMA(MODE_EMA)` (Wilder smoothing
for RSI/ATR, as MT5 uses).

In [ ]:
def rsi(close: pd.Series, period: int) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    out = 100 - (100 / (1 + rs))
    out[avg_loss == 0] = 100.0
    return out

def stochastic(high, low, close, k_period, d_period, slowing):
    lowest_low = low.rolling(k_period).min()
    highest_high = high.rolling(k_period).max()
    raw_k = 100 * (close - lowest_low) / (highest_high - lowest_low).replace(0, np.nan)
    k = raw_k.rolling(slowing).mean()
    d = k.rolling(d_period).mean()
    return k, d

def atr(high, low, close, period):
    prev_close = close.shift(1)
    tr = pd.concat([high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()

def ema(close, period):
    return close.ewm(span=period, min_periods=period, adjust=False).mean()

## 3. Fractal pivot detection

Direct port of `FindLastPivotLow` / `FindLastPivotHigh`.

In [ ]:
def find_pivots(series: pd.Series, left: int, right: int, mode: str) -> pd.Series:
    vals = series.values
    n = len(vals)
    is_pivot = np.zeros(n, dtype=bool)
    for i in range(left, n - right):
        wl = vals[i - left:i]
        wr = vals[i + 1:i + 1 + right]
        center = vals[i]
        if mode == "low":
            if np.all(wl >= center) and np.all(wr >= center):
                is_pivot[i] = True
        else:
            if np.all(wl <= center) and np.all(wr <= center):
                is_pivot[i] = True
    return pd.Series(is_pivot, index=series.index)

## 4. Divergence detection

Direct port of `CheckDivergence()`.

In [ ]:
def detect_divergence(price_low, price_high, osc, pivot_left, pivot_right,
                       max_pivot_age, min_gap, allow_hidden=False):
    piv_low = find_pivots(price_low, pivot_left, pivot_right, "low")
    piv_high = find_pivots(price_high, pivot_left, pivot_right, "high")
    low_idx = list(np.flatnonzero(piv_low.values))
    high_idx = list(np.flatnonzero(piv_high.values))

    out = pd.DataFrame(False, index=osc.index,
                        columns=["bullish", "bearish", "bullish_hidden", "bearish_hidden"])

    def pairs(idx_list):
        for a in range(len(idx_list)):
            for b in range(a + 1, len(idx_list)):
                i_old, i_new = idx_list[a], idx_list[b]
                if i_new - i_old < min_gap or i_new - i_old > max_pivot_age:
                    continue
                yield i_old, i_new

    for i_old, i_new in pairs(low_idx):
        pn, po = price_low.iloc[i_new], price_low.iloc[i_old]
        on, oo = osc.iloc[i_new], osc.iloc[i_old]
        if pn < po and on > oo:
            out.iloc[i_new, out.columns.get_loc("bullish")] = True
        elif allow_hidden and pn > po and on < oo:
            out.iloc[i_new, out.columns.get_loc("bullish_hidden")] = True

    for i_old, i_new in pairs(high_idx):
        pn, po = price_high.iloc[i_new], price_high.iloc[i_old]
        on, oo = osc.iloc[i_new], osc.iloc[i_old]
        if pn > po and on < oo:
            out.iloc[i_new, out.columns.get_loc("bearish")] = True
        elif allow_hidden and pn < po and on > oo:
            out.iloc[i_new, out.columns.get_loc("bearish_hidden")] = True

    return out

## 5. Price-action confirmation candle

Direct port of `IsBullishConfirmationCandle` / `IsBearishConfirmationCandle`.

In [ ]:
def bullish_confirmation(o, c, h, l, o_prev, c_prev, min_body_ratio, min_wick_ratio):
    rng = h - l
    if rng <= 0:
        return False
    body = abs(c - o)
    lower_wick = min(o, c) - l
    is_bull = c > o
    strong_body = is_bull and (body / rng >= min_body_ratio)
    engulfing = is_bull and (c > o_prev) and (o < c_prev) and (c_prev < o_prev)
    pin_bar = (lower_wick / rng >= min_wick_ratio) and (c > (l + rng * 0.5))
    return strong_body or engulfing or pin_bar

def bearish_confirmation(o, c, h, l, o_prev, c_prev, min_body_ratio, min_wick_ratio):
    rng = h - l
    if rng <= 0:
        return False
    body = abs(c - o)
    upper_wick = h - max(o, c)
    is_bear = c < o
    strong_body = is_bear and (body / rng >= min_body_ratio)
    engulfing = is_bear and (c < o_prev) and (o > c_prev) and (c_prev > o_prev)
    pin_bar = (upper_wick / rng >= min_wick_ratio) and (c < (h - rng * 0.5))
    return strong_body or engulfing or pin_bar

## 6. This desk's preset (Karat Trend)

Matches the `input` defaults shipped in `KaratTrend_Divergence_EA.mq5`.

In [ ]:
PRESET = {'pivot_left': 5, 'pivot_right': 5, 'max_pivot_age': 60, 'min_gap': 5, 'allow_hidden': False, 'rsi_period': 14, 'stoch_k': 14, 'stoch_d': 3, 'stoch_slowing': 3, 'trend_ema': 200, 'sl_atr_mult': 1.5, 'rr_ratio': 2.0, 'min_body_ratio': 0.55, 'min_wick_ratio': 0.55, 'confirm_timeout': 5, 'stoch_ob': 80, 'stoch_os': 20, 'atr_period': 14, 'require_both': True}
PRESET

## 7. Build signals

In [ ]:
def build_signals(df: pd.DataFrame, p: dict) -> pd.DataFrame:
    out = df.copy()
    out["rsi"] = rsi(out["close"], p["rsi_period"])
    out["stoch_k"], out["stoch_d"] = stochastic(out["high"], out["low"], out["close"],
                                                 p["stoch_k"], p["stoch_d"], p["stoch_slowing"])
    out["atr"] = atr(out["high"], out["low"], out["close"], p["atr_period"])
    out["trend_ema"] = ema(out["close"], p["trend_ema"])

    rsi_div = detect_divergence(out["low"], out["high"], out["rsi"],
                                 p["pivot_left"], p["pivot_right"], p["max_pivot_age"],
                                 p["min_gap"], p["allow_hidden"])
    stoch_div = detect_divergence(out["low"], out["high"], out["stoch_k"],
                                   p["pivot_left"], p["pivot_right"], p["max_pivot_age"],
                                   p["min_gap"], p["allow_hidden"])

    if p["require_both"]:
        bullish = rsi_div["bullish"] & stoch_div["bullish"]
        bearish = rsi_div["bearish"] & stoch_div["bearish"]
    else:
        bullish = rsi_div["bullish"] | stoch_div["bullish"]
        bearish = rsi_div["bearish"] | stoch_div["bearish"]

    out["div_bullish"] = bullish.fillna(False)
    out["div_bearish"] = bearish.fillna(False)
    return out

sig = build_signals(df, PRESET)
sig[["time", "close", "rsi", "stoch_k", "div_bullish", "div_bearish"]].tail(10)

## 8. Simplified backtest loop

Mirrors `OnTick()`'s state machine: on a divergence, arm a confirmation watcher; on a
confirming candle within the timeout, enter with an ATR-based stop and R-multiple
target; manage one position at a time. Simplifications vs the EA: no spread/slippage,
no daily-loss lock, no trailing/breakeven (fixed SL/TP to first touch).

In [ ]:
def simple_backtest(sig: pd.DataFrame, p: dict, trend_filter=True):
    trades = []
    waiting = None
    position = None

    o, h, l, c = sig["open"].values, sig["high"].values, sig["low"].values, sig["close"].values
    atr_v = sig["atr"].values
    ema_v = sig["trend_ema"].values
    stoch_k = sig["stoch_k"].values

    for i in range(2, len(sig)):
        if position is not None:
            if position["bullish"]:
                if l[i] <= position["sl"]:
                    trades.append({**position, "exit": position["sl"], "exit_i": i, "win": False})
                    position = None
                elif h[i] >= position["tp"]:
                    trades.append({**position, "exit": position["tp"], "exit_i": i, "win": True})
                    position = None
            else:
                if h[i] >= position["sl"]:
                    trades.append({**position, "exit": position["sl"], "exit_i": i, "win": False})
                    position = None
                elif l[i] <= position["tp"]:
                    trades.append({**position, "exit": position["tp"], "exit_i": i, "win": True})
                    position = None
            continue

        if waiting is not None:
            confirmed = (bullish_confirmation(o[i-1], c[i-1], h[i-1], l[i-1], o[i-2], c[i-2],
                                               p["min_body_ratio"], p["min_wick_ratio"])
                         if waiting["bullish"] else
                         bearish_confirmation(o[i-1], c[i-1], h[i-1], l[i-1], o[i-2], c[i-2],
                                               p["min_body_ratio"], p["min_wick_ratio"]))
            if confirmed:
                bullish = waiting["bullish"]
                entry = c[i]
                a = atr_v[i]
                if not np.isnan(a) and a > 0:
                    if bullish:
                        sl = min(waiting["pivot_price"], l[i-1]) - a * p["sl_atr_mult"]
                        risk = entry - sl
                        tp = entry + risk * p["rr_ratio"]
                    else:
                        sl = max(waiting["pivot_price"], h[i-1]) + a * p["sl_atr_mult"]
                        risk = sl - entry
                        tp = entry - risk * p["rr_ratio"]
                    if risk > 0:
                        position = {"bullish": bullish, "entry": entry, "sl": sl, "tp": tp, "entry_i": i}
                waiting = None
            else:
                waiting["bars_left"] -= 1
                if waiting["bars_left"] <= 0:
                    waiting = None
            continue

        if not np.isnan(ema_v[i]):
            bullish_ok = c[i] > ema_v[i]
            if trend_filter:
                allow_bull = bullish_ok or stoch_k[i] <= p["stoch_os"]
                allow_bear = (not bullish_ok) or stoch_k[i] >= p["stoch_ob"]
            else:
                allow_bull = allow_bear = True

            if sig["div_bullish"].iloc[i] and allow_bull:
                waiting = {"bullish": True, "bars_left": p["confirm_timeout"], "pivot_price": l[i]}
            elif sig["div_bearish"].iloc[i] and allow_bear:
                waiting = {"bullish": False, "bars_left": p["confirm_timeout"], "pivot_price": h[i]}

    return pd.DataFrame(trades)

trades = simple_backtest(sig, PRESET)
print(f"{len(trades)} trades")
if len(trades):
    print(f"Win rate: {trades['win'].mean():.1%}")
trades.tail()

## 9. Quick equity curve (R-multiples)

In [ ]:
def r_multiple_curve(trades, rr_ratio):
    if trades.empty:
        print("No trades to plot.")
        return
    r = np.where(trades["win"], rr_ratio, -1.0)
    equity = np.cumsum(r)
    plt.figure(figsize=(9, 4))
    plt.plot(equity, marker="o", markersize=3)
    plt.axhline(0, color="grey", lw=0.8)
    plt.title("Karat Trend — cumulative R-multiples (" + str(len(trades)) + " trades)")
    plt.xlabel("Trade #")
    plt.ylabel("Cumulative R")
    plt.grid(alpha=0.3)
    plt.show()
    print(f"Total R: {equity[-1]:.2f}  |  Expectancy per trade: {equity[-1] / len(trades):.3f} R")

r_multiple_curve(trades, PRESET["rr_ratio"])

## Next steps

- Swap in your real MT5 export at `DATA_PATH`.
- Sweep `pivot_left/right`, `max_pivot_age`, and RSI/Stoch periods and compare
  `expectancy_R` -- faster here than MT5's optimizer for signal-quality triage.
- Translate any promising parameter region back into this file's `.mq5` `input`
  defaults and confirm with a real Strategy Tester run (spread, slippage, daily-loss
  lock, and trailing/breakeven all matter and are not modeled here) -- see
  `BACKTEST.md`.